In [1]:
import kagglehub
path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")

Using Colab cache for faster access to the 'international-football-results-from-1872-to-2017' dataset.


In [2]:
import os
import pandas as pd

# List the contents of the downloaded directory
print(os.listdir(path))

# Assuming the main data file is 'results.csv' or similar
# Adjust this if the file name is different after inspecting the output of os.listdir(path)
data_file_path = os.path.join(path, 'results.csv')

# Load the dataset
df = pd.read_csv(data_file_path)

# Display the first few rows of the DataFrame
display(df.head())

['former_names.csv', 'goalscorers.csv', 'shootouts.csv', 'results.csv']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [3]:
# Display basic information about the DataFrame (data types, non-null counts)
display(df.info())

# Display descriptive statistics for numerical columns
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49547 entries, 0 to 49546
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        49547 non-null  object
 1   home_team   49547 non-null  object
 2   away_team   49547 non-null  object
 3   home_score  49547 non-null  int64 
 4   away_score  49547 non-null  int64 
 5   tournament  49547 non-null  object
 6   city        49547 non-null  object
 7   country     49547 non-null  object
 8   neutral     49547 non-null  bool  
dtypes: bool(1), int64(2), object(6)
memory usage: 3.1+ MB


None

,home_score,away_score
count,49547.000000,49547.000000
mean,1.757140,1.182715
std,1.773702,1.402353
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,1.000000
75%,2.000000,2.000000
max,31.000000,21.000000


In [4]:
# Convert 'date' column to datetime objects
df['date'] = pd.to_datetime(df['date'])

# Extract year and month as potential features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# Define the target variable: match outcome
# 0: Draw, 1: Home Win, 2: Away Win
def get_match_outcome(row):
    if row['home_score'] == row['away_score']:
        return 0  # Draw
    elif row['home_score'] > row['away_score']:
        return 1  # Home Win
    else:
        return 2  # Away Win

df['outcome'] = df.apply(get_match_outcome, axis=1)

# Display the updated DataFrame with new features and target variable
display(df.head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,month,outcome
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1872,11,0
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1873,3,1
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1874,3,1
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1875,3,0
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1876,3,1


In [5]:
# Check the distribution of the target variable
display(df['outcome'].value_counts())

,count
outcome,
1,24276
2,14010
0,11261


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define categorical and numerical features
categorical_features = ['home_team', 'away_team', 'tournament', 'city', 'country']
numerical_features = ['home_score', 'away_score', 'year', 'month', 'neutral']

# Define the target variable
y = df['outcome']

# Features (X) will include numerical and categorical features
X = df[numerical_features + categorical_features]

# Create a column transformer for one-hot encoding categorical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # Keep numerical features as they are
)

# Split the data into training and test sets first
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full) # 0.25 * 0.8 = 0.2

print(f"Original dataset shape: {X.shape}")
print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")

# Apply the preprocessing pipeline to the features
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed Training set shape: {X_train_processed.shape}")
print(f"Processed Validation set shape: {X_val_processed.shape}")
print(f"Processed Test set shape: {X_test_processed.shape}")

Original dataset shape: (49547, 10)
Training set shape: (29727, 10)
Validation set shape: (9910, 10)
Test set shape: (9910, 10)
Processed Training set shape: (29727, 2922)
Processed Validation set shape: (9910, 2922)
Processed Test set shape: (9910, 2922)


In [9]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import scipy.sparse # Import scipy.sparse

# Convert sparse matrices to dense arrays for Keras, if they are sparse
# Keras prefers dense arrays for input to its layers
if isinstance(X_train_processed, (scipy.sparse.csr.csr_matrix, scipy.sparse.csc.csc_matrix)):
    X_train_processed = X_train_processed.toarray()
    X_val_processed = X_val_processed.toarray()
    X_test_processed = X_test_processed.toarray()

# Get the number of features after one-hot encoding
input_shape = X_train_processed.shape[1]

# Define the deep learning model
model = keras.Sequential([
    layers.Input(shape=(input_shape,)),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax') # 3 classes for outcome: Draw, Home Win, Away Win
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display model summary
model.summary()

/tmp/ipykernel_1264/3614944325.py:8: DeprecationWarning: Please import `csr_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csr` namespace is deprecated and will be removed in SciPy 2.0.0.
  if isinstance(X_train_processed, (scipy.sparse.csr.csr_matrix, scipy.sparse.csc.csc_matrix)):
/tmp/ipykernel_1264/3614944325.py:8: DeprecationWarning: Please import `csc_matrix` from the `scipy.sparse` namespace; the `scipy.sparse.csc` namespace is deprecated and will be removed in SciPy 2.0.0.
  if isinstance(X_train_processed, (scipy.sparse.csr.csr_matrix, scipy.sparse.csc.csc_matrix)):


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       748,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 789,635 (3.01 MB)

 Trainable params: 789,635 (3.01 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Train the model
history = model.fit(
    X_train_processed,
    y_train,
    epochs=10, # You can adjust the number of epochs
    batch_size=32,
    validation_data=(X_val_processed, y_val)
)

# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test_processed, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

Epoch 1/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 21s 21ms/step - accuracy: 0.4537 - loss: 1.6781 - val_accuracy: 0.4899 - val_loss: 1.0435
Epoch 2/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 14s 14ms/step - accuracy: 0.4884 - loss: 1.0468 - val_accuracy: 0.4899 - val_loss: 1.0436
Epoch 3/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.4897 - loss: 1.0448 - val_accuracy: 0.4899 - val_loss: 1.0435
Epoch 4/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 14s 15ms/step - accuracy: 0.4898 - loss: 1.0442 - val_accuracy: 0.4899 - val_loss: 1.0435
Epoch 5/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.4897 - loss: 1.0443 - val_accuracy: 0.4899 - val_loss: 1.0435
Epoch 6/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.4895 - loss: 1.0448 - val_accuracy: 0.4899 - val_loss: 1.0436
Epoch 7/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.4896 - loss: 1.0444 - val_accuracy: 0.4899 - val_loss: 1.0437
Epoch 8/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.4898 - loss: 1.0447 - 

In [11]:
# Install Streamlit
!pip install streamlit joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 51.6 MB/s eta 0:00:00


In [12]:
import joblib

# Save the preprocessor
joblib.dump(preprocessor, 'preprocessor.pkl')

# Save the Keras model
model.save('football_match_predictor_model.h5')

print("Preprocessor and model saved successfully.")

Preprocessor and model saved successfully.


In [13]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from tensorflow import keras

# Load the preprocessor and model
preprocessor = joblib.load('preprocessor.pkl')
model = keras.models.load_model('football_match_predictor_model.h5')

st.title('International Football Match Outcome Predictor')
st.write('Enter match details to predict the outcome (Home Win, Draw, Away Win).')

# Input fields for match details
home_team = st.text_input('Home Team', 'England')
away_team = st.text_input('Away Team', 'Scotland')
home_score_input = st.number_input('Home Score (for prediction purposes, use recent average)', min_value=0, max_value=20, value=1)
away_score_input = st.number_input('Away Score (for prediction purposes, use recent average)', min_value=0, max_value=20, value=1)
year = st.slider('Year', min_value=1872, max_value=2026, value=2024)
month = st.slider('Month', min_value=1, max_value=12, value=7)
tournament = st.text_input('Tournament', 'Friendly')
city = st.text_input('City', 'London')
country = st.text_input('Country', 'England')
neutral = st.checkbox('Neutral Venue', False)

if st.button('Predict Outcome'):
    # Create a DataFrame from user input
    input_data = pd.DataFrame([{
        'home_team': home_team,
        'away_team': away_team,
        'home_score': home_score_input,
        'away_score': away_score_input,
        'year': year,
        'month': month,
        'tournament': tournament,
        'city': city,
        'country': country,
        'neutral': neutral
    }])

    # Preprocess the input data
    processed_input = preprocessor.transform(input_data)

    # Make prediction
    prediction_proba = model.predict(processed_input)
    predicted_class = np.argmax(prediction_proba, axis=1)[0]

    outcome_map = {0: 'Draw', 1: 'Home Win', 2: 'Away Win'}
    predicted_outcome = outcome_map[predicted_class]

    st.subheader('Prediction:')
    st.write(f"The predicted outcome is: **{predicted_outcome}**")
    st.write(f"Confidence (Draw, Home Win, Away Win): {prediction_proba[0]}")


Writing app.py


To run the Streamlit app, execute the following commands in a new cell. After execution, a local URL will be provided that you can click to access the app. It might take a moment for the app to start up and the URL to appear.

```bash
!streamlit run app.py &>/dev/null&
import subprocess
subprocess.Popen(['npx', 'localtunnel', '--port', '8501'])
```


In [14]:
%%writefile app.py

import streamlit as st
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf
from tensorflow import keras

# Load the preprocessor and model
preprocessor = joblib.load('preprocessor.pkl')
model = keras.models.load_model('football_match_predictor_model.h5')

st.title('⚽ International Football Match Outcome Predictor 🥅')
st.write('Enter match details to predict the outcome (Home Win, Draw, Away Win).')

st.sidebar.header('Match Details Input 👇')

# Input fields for match details in the sidebar
home_team = st.sidebar.text_input('🏠 Home Team', 'England')
away_team = st.sidebar.text_input('✈️ Away Team', 'Scotland')
home_score_input = st.sidebar.number_input('🥅 Home Score (avg. or guess)', min_value=0, max_value=20, value=1)
away_score_input = st.sidebar.number_input('🥅 Away Score (avg. or guess)', min_value=0, max_value=20, value=1)
year = st.sidebar.slider('🗓️ Year', min_value=1872, max_value=2026, value=2024)
month = st.sidebar.slider('🗓️ Month', min_value=1, max_value=12, value=7)
tournament = st.sidebar.text_input('🏆 Tournament', 'Friendly')
city = st.sidebar.text_input('🏙️ City', 'London')
country = st.sidebar.text_input('🌍 Country', 'England')
neutral = st.sidebar.checkbox('Neutral Venue', False)

if st.button('🚀 Predict Outcome'):
    # Create a DataFrame from user input
    input_data = pd.DataFrame([{
        'home_team': home_team,
        'away_team': away_team,
        'home_score': home_score_input,
        'away_score': away_score_input,
        'year': year,
        'month': month,
        'tournament': tournament,
        'city': city,
        'country': country,
        'neutral': neutral
    }])

    # Preprocess the input data
    processed_input = preprocessor.transform(input_data)

    # Make prediction
    prediction_proba = model.predict(processed_input)
    predicted_class = np.argmax(prediction_proba, axis=1)[0]

    outcome_map = {0: 'Draw', 1: 'Home Win', 2: 'Away Win'}
    predicted_outcome = outcome_map[predicted_class]

    st.subheader('Prediction Results 📊:')
    st.write(f"The predicted outcome is: **{predicted_outcome}**")
    st.write(f"Confidence (Draw, Home Win, Away Win): {prediction_proba[0]}")


Overwriting app.py


The `app.py` file has been updated with the input fields in the sidebar and added emojis. Now, you need to re-run the Streamlit app to see the changes. Execute the following commands in a new cell:

```bash
!streamlit run app.py &>/dev/null&
import subprocess
subprocess.Popen(['npx', 'localtunnel', '--port', '8501'])
```

In [15]:
!pip install --ignore-installed blinker
!pip install streamlit

In [16]:
!pip install streamlit pyngrok

In [17]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [18]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [19]:
# Run the Streamlit app
# This will provide a public URL to access the app
!streamlit run app.py &




2026-08-28 15:43:48.237 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.190.128.184:8501

2026-08-28 15:44:10.253311: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step
  Stopping...
